# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df





In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock [mm]": 170.0,
    "rear_shock [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Parse csv path ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

print(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    print("  ", p.name)

sessions = []
events_all = []
metrics_all = []

for p in CSV_FILES:
    print(f"\nProcessing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
    )

    session = results["session"]
    src = p.stem
    session["meta"]["source_file"] = src
    sessions.append(session)

    if results.get("events") is not None:
        events_all.append(results["events"].assign(source_file=src))

    if results.get("metrics") is not None:
        metrics_all.append(results["metrics"].assign(source_file=src))

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

print("\nDone.")
print(f"Sessions: {len(sessions)}")
print(f"Events rows: {len(events_df)}")
print(f"Metrics rows: {len(metrics_df)}")


Found 2 files:
   2026-01-04_06-37-32.CSV
   2026-01-04_07-08-37.CSV

Processing 2026-01-04_06-37-32.CSV ...
[DEBUG] detect_events_from_schema: meta keys: ['channels', 'channel_info', 'sample_rate_hz', 'sample_rate_by_channel_hz', 'device', 'notes', 'streams', 'signals']
[DEBUG] detect_events_from_schema: meta.signals present: True len= 12
[DEBUG] deep_compression(rear_shock): raw candidates=1042
[DEBUG] deep_compression(rear_shock): after debounce=337 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] deep_compression(rear_shock): passed conditions=1
[DEBUG] deep_compression(front_shock): raw candidates=1472
[DEBUG] deep_compression(front_shock): after debounce=235 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] deep_compression(front_shock): passed conditions=0
[DEBUG] rebounds(rear_shock): raw candidates=4205
[DEBUG] rebounds(rear_shock): after debounce=32 (gap=0.3s, prefer_key=disp, prefer_abs=False)
[DEBUG] rebounds(rear_shock): passed conditions=0
[DEBUG] rebounds(front_